# TD3 — Analyse d'un programme de dessalement au Maroc

## Présentation
Ce notebook utilise une petite table de projets pour pratiquer la création de structures de données, le parsing CSV, les boucles, la mise à jour d'objets, les statistiques simples et les tests.

> **Important :** les lignes ci-dessous sont les données fournies dans le TP. Elles sont utilisées comme jeu de données pédagogique et ne sont pas présentées ici comme une base officielle ou vérifiée.

## 1. Structure des données

In [ ]:
from dataclasses import dataclass
import csv
from io import StringIO

@dataclass
class DesalinationProject:
    name: str
    location: str
    capacity_million_m3: float
    status: str
    partners: list[str]

## 2. Données et lecture CSV

In [ ]:
RAW_DATA = """name,location,capacity_million_m3,status,partners
Agadin Desalination Plant,Agadin,275,Operational,Cox;AMEA Power
Casablanca Mega-Plant,Casablanca,548,Under construction,ACCIONA;Green of Africa;AfriquiaGaz
Dakhla Atlantic Plant,Dakhla,300,Under construction,DAMEC;Ministry of Energy
Labyoune Extension,Labyoune,200,Funding secured,AMEA Power;Government of Morocco
Safi Industrial Plant,Safi,150,Operational,OCP Group
Jorf L safar Desal,Jorf L safar,100,Industrial,OCP Group
"""

def load_projects(raw_data: str) -> list[DesalinationProject]:
    projects = []
    reader = csv.DictReader(StringIO(raw_data.strip()))
    required = {"name", "location", "capacity_million_m3", "status", "partners"}

    if not required.issubset(reader.fieldnames or []):
        raise ValueError("Le fichier ne contient pas toutes les colonnes attendues.")

    for row in reader:
        capacity = float(row["capacity_million_m3"])
        if capacity < 0:
            raise ValueError("La capacité ne peut pas être négative.")
        partners = [p.strip() for p in row["partners"].split(";") if p.strip()]
        projects.append(DesalinationProject(
            name=row["name"].strip(),
            location=row["location"].strip(),
            capacity_million_m3=capacity,
            status=row["status"].strip(),
            partners=partners
        ))
    return projects


projects = load_projects(RAW_DATA)
print(f"{len(projects)} projets chargés.")

## 3. Exploration et mise à jour des projets

In [ ]:
print("=== LISTE DES PROJETS ===")
for project in projects:
    print(f"{project.name} | {project.location} | {project.status}")

# Conserver une copie des capacités initiales pour comparer avant/après
original_capacities = {
    project.name: project.capacity_million_m3 for project in projects
}

# Augmentation hypothétique de 15 % pour les projets non opérationnels
for project in projects:
    if project.status in ["Under construction", "Funding secured"]:
        project.capacity_million_m3 = round(project.capacity_million_m3 * 1.15, 1)

# Mise à jour du statut du projet de Casablanca, conformément au scénario du TP
for project in projects:
    if "Casablanca" in project.name:
        project.status = "Inauguration 2026"
        break

print("\nCapacités initiales et après scénario :")
for project in projects:
    print(
        f"{project.name}: {original_capacities[project.name]} "
        f"→ {project.capacity_million_m3} millions de m³"
    )

## 4. Analyse des partenaires et indicateurs

In [ ]:
unique_partners = sorted({
    partner
    for project in projects
    for partner in project.partners
})

total_capacity = sum(p.capacity_million_m3 for p in projects)
operational_count = sum(
    p.status in ["Operational", "Industrial"] for p in projects
)
largest_project = max(projects, key=lambda p: p.capacity_million_m3)

total_billion = round(total_capacity / 1000, 2)

print("Nombre de projets :", len(projects))
print(f"Capacité totale : {total_capacity:.1f} millions de m³")
print(f"Capacité totale : {total_billion:.2f} milliard(s) de m³")
print("Projets opérationnels/industriels :", operational_count)
print(
    "Projet de plus grande capacité :",
    largest_project.name,
    f"({largest_project.capacity_million_m3} millions de m³)"
)
print("Partenaires uniques :", ", ".join(unique_partners))

## 5. Vérification des résultats

In [ ]:
assert len(projects) == 6
casablanca = next(p for p in projects if "Casablanca" in p.name)
assert casablanca.capacity_million_m3 == 630.2
assert operational_count == 3
assert total_billion == 1.73

print("Tous les tests ont réussi.")

## 6. Conclusion

Ce TP montre comment transformer des données textuelles en objets Python, parcourir et modifier une collection, puis produire des indicateurs synthétiques.

**Compétences mobilisées :** `dataclass`, CSV, parsing, listes, ensembles, boucles, fonctions intégrées (`sum`, `max`), assertions et statistiques descriptives simples.

**Limite à signaler dans le README :** les capacités modifiées de 15 % et le changement de statut sont des hypothèses de scénario du TP, pas des prévisions vérifiées.